# Covarion: базовые геодезические сети

В этом ноутбуке рассмотрены простые модели геодезических построений
и получение априорных ковариационных матриц координатных параметров.

Рассматриваются четыре примера:

1. Линейная засечка по наклонным расстояниям.
2. Полярное определение точки по азимуту и расстоянию.
3. Полярное определение точки через установку электронного тахеометра.
4. Угловая засечка с одной станции тахеометра.

Во всех примерах Covarion строит матрицу коэффициентов наблюдений
и ковариационную матрицу параметров:

$$
\mathbf{C}_{\hat{x}} =
\left(
\mathbf{A}^{\mathsf{T}}
\mathbf{C}_l^{-1}
\mathbf{A}
\right)^{-1}.
$$

Для жёстко закреплённых исходных пунктов используются точные
датумные ограничения, а не нулевая ковариационная матрица наблюдений.

In [1]:
from __future__ import annotations

import math

import numpy as np

from covarion import (
    AzimuthObservation,
    ControlPointObservation,
    GeodeticNetwork,
    GeodeticPoint,
    ObservationCovarianceMethod,
    SlopeDistanceObservation,
    TotalStationSetup,
    TotalStationSight,
)

In [2]:
def arcseconds_to_radians(value: float) -> float:
    """Convert angular standard deviation from arcseconds to radians."""
    return math.radians(value / 3600.0)


def make_point(
    name: str,
    x: float,
    y: float,
) -> GeodeticPoint:
    """Create a simple 2D point with placeholder local covariance."""
    return GeodeticPoint(
        name=name,
        coordinates=(x, y),
        axes=("X", "Y"),
        covariance=np.eye(2),
    )


def make_enh_point(
    name: str,
    east: float,
    north: float,
    height: float,
) -> GeodeticPoint:
    """Create a simple 3D ENH point with placeholder local covariance."""
    return GeodeticPoint(
        name=name,
        coordinates=(east, north, height),
        axes=("E", "N", "H"),
        covariance=np.eye(3),
    )


def print_covariance_summary(covariance) -> None:
    """Print main covariance diagnostics in a compact form."""
    print(f"Method: {covariance.method_name}")
    print(f"Dimension: {covariance.dimension}")
    print(f"Datum: {covariance.metadata.get('datum_kind')}")
    print(
        "Minimum eigenvalue: "
        f"{covariance.minimum_eigenvalue:.3e}"
    )
    print()

    print("Standard deviations:")
    for name, sigma in covariance.standard_deviations.items():
        print(f"  {name:>8s}: {sigma:.6f}")

    print()
    print("Covariance matrix:")
    print(covariance.matrix)

## 1. Линейная засечка по расстояниям

Пусть известны два исходных пункта:

$$
A=(0, 0),
\qquad
B=(100, 0).
$$

Определяемый пункт:

$$
P=(40, 60).
$$

Измеряются расстояния:

$$
s_{AP},
\qquad
s_{BP}.
$$

Пункты $A$ и $B$ закрепляются жёсткими координатными ограничениями,
а координаты точки $P$ определяются двумя расстояниями.

В такой плоской постановке две дальности дают две независимые
геометрические связи для неизвестных $X_P$ и $Y_P$.

In [3]:
linear_intersection_network = GeodeticNetwork(
    name="Linear intersection",
    points=(
        make_point("A", 0.0, 0.0),
        make_point("B", 100.0, 0.0),
        make_point("P", 40.0, 60.0),
    ),
)

In [4]:
linear_intersection_method = ObservationCovarianceMethod(
    observations=(
        # Жёстко фиксируем координаты исходных пунктов A и B.
        ControlPointObservation.fixed(
            name="A_fixed",
            point_name="A",
            axes=("X", "Y"),
            coordinates=(0.0, 0.0),
        ),
        ControlPointObservation.fixed(
            name="B_fixed",
            point_name="B",
            axes=("X", "Y"),
            coordinates=(100.0, 0.0),
        ),

        # Измеренные наклонные расстояния.
        # В 2D-сети SlopeDistanceObservation эквивалентен
        # обычному плановому расстоянию.
        SlopeDistanceObservation(
            name="s_AP",
            from_point="A",
            to_point="P",
            constant_error=0.003,
            ppm_error=2.0,
        ),
        SlopeDistanceObservation(
            name="s_BP",
            from_point="B",
            to_point="P",
            constant_error=0.003,
            ppm_error=2.0,
        ),
    ),
)

In [5]:
linear_intersection_covariance = (
    linear_intersection_network.compute_covariance(
        linear_intersection_method
    )
)

print_covariance_summary(linear_intersection_covariance)

Method: linearized-observations
Dimension: 6
Datum: fixed-control
Minimum eigenvalue: -3.487e-22

Standard deviations:
       A_X: 0.000000
       A_Y: -0.000000
       B_X: 0.000000
       B_Y: 0.000000
       P_X: 0.003345
       P_Y: 0.002753

Covariance matrix:
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   5.97691757e-22  1.21819349e-22]
 [ 0.00000000e+00 -0.00000000e+00  0.00000000e+00  0.00000000e+00
   3.24851596e-22  6.42099869e-22]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   6.32511942e-22 -3.69886222e-22]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  -3.69886222e-22  1.31312860e-22]
 [ 5.97691757e-22  3.24851596e-22  6.32511942e-22 -3.69886222e-22
   1.11915520e-05  3.56992000e-07]
 [ 1.21819349e-22  6.42099869e-22 -3.69886222e-22  1.31312860e-22
   3.56992000e-07  7.58003200e-06]]


In [6]:
p_covariance = linear_intersection_covariance.diagonal_block("P")

print("Local covariance block of point P:")
print(p_covariance)

print()
print("Point P standard deviations:")
print(
    "sigma_XP = "
    f"{linear_intersection_covariance.standard_deviations['P_X']:.6f}"
)
print(
    "sigma_YP = "
    f"{linear_intersection_covariance.standard_deviations['P_Y']:.6f}"
)

Local covariance block of point P:
[[1.1191552e-05 3.5699200e-07]
 [3.5699200e-07 7.5800320e-06]]

Point P standard deviations:
sigma_XP = 0.003345
sigma_YP = 0.002753


## 2. Полярное определение точки по азимуту и расстоянию

Пусть известна станция:

$$
S=(0, 0).
$$

Определяемая точка:

$$
P=(60, 80).
$$

Измеряются:

- абсолютный азимут $\alpha_{SP}$, отсчитанный по часовой стрелке от
  северного направления;
- расстояние $s_{SP}$.

Такой пример подходит только для азимута, уже ориентированного
во внешней системе координат: например, после GNSS-ориентирования,
гиротеодолитного определения азимута или иной процедуры задания севера.

В отличие от отсчёта по лимбу тахеометра, здесь не нужна общая
поправка ориентировки станции.

In [7]:
polar_network = GeodeticNetwork(
    name="Polar point determination",
    points=(
        make_enh_point("S", 0.0, 0.0, 0.0),
        make_enh_point("P", 60.0, 80.0, 0.0),
    ),
)

In [9]:
polar_method = ObservationCovarianceMethod(
    observations=(
        # Фиксируем станцию S во внешней системе координат.
        ControlPointObservation.fixed(
            name="S_fixed",
            point_name="S",
            axes=("E", "N", "H"),
            coordinates=(0.0, 0.0, 0.0),
        ),

        # Абсолютный азимут от севера.
        AzimuthObservation(
            name="az_SP",
            from_point="S",
            to_point="P",
            standard_deviation=arcseconds_to_radians(2.0),
        ),

        # Пространственная длина линии S-P.
        SlopeDistanceObservation(
            name="s_SP",
            from_point="S",
            to_point="P",
            constant_error=0.002,
            ppm_error=2.0,
        ),

        # В этом примере высота P фиксируется только для того,
        # чтобы устранить вертикальную неопределённость:
        # азимут и горизонтальное расстояние сами по себе не определяют H.
        ControlPointObservation.fixed(
            name="P_height_fixed",
            point_name="P",
            axes=("H",),
            coordinates=(0.0,),
        ),
    ),
)

In [10]:
polar_covariance = polar_network.compute_covariance(polar_method)

print_covariance_summary(polar_covariance)

Method: linearized-observations
Dimension: 6
Datum: fixed-control
Minimum eigenvalue: -3.422e-22

Standard deviations:
       S_E: 0.000000
       S_N: 0.000000
       S_H: 0.000000
       P_E: 0.001434
       P_N: 0.001710
       P_H: 0.000000

Covariance matrix:
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 2.67244754e-23
  2.93476404e-23 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.67244754e-23 0.00000000e+00 0.00000000e+00 2.05611342e-06
  1.48791493e-06 0.00000000e+00]
 [2.93476404e-23 0.00000000e+00 0.00000000e+00 1.48791493e-06
  2.92406380e-06 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]


В этой модели координаты точки $P$ определяются:

$$
\alpha_{SP} = \operatorname{atan2}(\Delta E, \Delta N),
$$

$$
s_{SP} =
\sqrt{
\Delta E^2 +
\Delta N^2 +
\Delta H^2
}.
$$

Высота $H_P$ закреплена отдельным ограничением, потому что
азимут не содержит вертикальной информации, а при горизонтальной линии
наклонное расстояние имеет нулевую производную по высоте первого порядка.

## 3. Определение точки через установку электронного тахеометра

Здесь рассматривается одна установка тахеометра на пункте $S$.

Из станции выполняются визирования:

- на исходную точку `A`;
- на исходную точку `B`;
- на определяемую точку `P`.

Каждая визура содержит:

- горизонтальный отсчёт;
- зенитный угол;
- наклонное расстояние.

Горизонтальные отсчёты имеют общий неизвестный ноль лимба.
Covarion исключает его преобразованием направлений в угловые разности.

Если наблюдены $m$ горизонтальных направлений, после исключения
ориентировки остаётся:

$$
m-1
$$

независимых угловых уравнений.

In [11]:
total_station_network = GeodeticNetwork(
    name="Total station polar network",
    points=(
        make_enh_point("S", 20.0, 20.0, 10.0),
        make_enh_point("A", 0.0, 120.0, 12.0),
        make_enh_point("B", 120.0, 0.0, 11.0),
        make_enh_point("P", 75.0, 70.0, 14.0),
    ),
)

In [12]:
total_station_setup = TotalStationSetup(
    name="S_setup_01",
    station="S",
    reference_target="A",
    sights=(
        # Опорная визура. После исключения общего нуля лимба
        # все горизонтальные направления будут приведены к A.
        TotalStationSight(
            target="A",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
            zenith_standard_deviation=arcseconds_to_radians(2.0),
            slope_distance_standard_deviation=0.002,
        ),
        TotalStationSight(
            target="B",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
            zenith_standard_deviation=arcseconds_to_radians(2.0),
            slope_distance_standard_deviation=0.002,
        ),
        TotalStationSight(
            target="P",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
            zenith_standard_deviation=arcseconds_to_radians(2.0),
            slope_distance_standard_deviation=0.003,
        ),
    ),
)

In [13]:
total_station_linearized = total_station_setup.linearize(
    total_station_network
)

print("Design matrix shape:")
print(total_station_linearized.design_matrix.shape)

print()
print("Observation covariance shape:")
print(total_station_linearized.covariance.shape)

print()
print("Labels:")
for label in total_station_linearized.labels:
    print(f"  {label}")

print()
print("Local observation covariance matrix:")
print(total_station_linearized.covariance)

Design matrix shape:
(8, 12)

Observation covariance shape:
(8, 8)

Labels:
  S_setup_01:A->B:horizontal-angle
  S_setup_01:A->P:horizontal-angle
  S_setup_01:A:zenith-angle
  S_setup_01:B:zenith-angle
  S_setup_01:P:zenith-angle
  S_setup_01:A:slope-distance
  S_setup_01:B:slope-distance
  S_setup_01:P:slope-distance

Local observation covariance matrix:
[[1.88035444e-10 9.40177222e-11 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.40177222e-11 1.88035444e-10 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 9.40177222e-11 0.00000000e+00
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 9.40177222e-11
  0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  9.40177222e-11 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 

In [14]:
total_station_method = ObservationCovarianceMethod(
    observations=(
        # Исходные точки A и B задают внешний датум сети.
        ControlPointObservation.fixed(
            name="A_fixed",
            point_name="A",
            axes=("E", "N", "H"),
            coordinates=(0.0, 120.0, 12.0),
        ),
        ControlPointObservation.fixed(
            name="B_fixed",
            point_name="B",
            axes=("E", "N", "H"),
            coordinates=(120.0, 0.0, 11.0),
        ),

        # Для примера фиксируем координаты станции S.
        # В реальной задаче S может быть известной, определяемой
        # или слабо закреплённой стохастическим контролем.
        ControlPointObservation.fixed(
            name="S_fixed",
            point_name="S",
            axes=("E", "N", "H"),
            coordinates=(20.0, 20.0, 10.0),
        ),

        total_station_setup,
    ),
)

total_station_covariance = total_station_network.compute_covariance(
    total_station_method
)

print_covariance_summary(total_station_covariance)

Method: linearized-observations
Dimension: 12
Datum: fixed-control
Minimum eigenvalue: -2.238e-21

Standard deviations:
       S_E: 0.000000
       S_N: 0.000000
       S_H: 0.000000
       A_E: 0.000000
       A_N: -0.000000
       A_H: 0.000000
       B_E: 0.000000
       B_N: 0.000000
       B_H: 0.000000
       P_E: 0.002295
       P_N: 0.002118
       P_H: 0.000739

Covariance matrix:
[[ 0.00000000e+00 -2.72161702e-22 -3.76986835e-24 -9.43696857e-22
  -1.19204235e-21 -6.66546190e-23 -1.54518143e-21 -1.36863068e-22
  -4.51636778e-23  1.59105118e-22  1.29338563e-22 -5.75224111e-23]
 [-2.72161702e-22  0.00000000e+00 -2.41689525e-23  3.27004876e-22
  -8.45730526e-23 -1.92295367e-23  3.37121830e-22  2.07168883e-22
  -1.01268114e-24  1.63475350e-21 -1.18821127e-22 -3.72907931e-23]
 [-3.76986835e-24 -2.41689525e-23  0.00000000e+00 -1.38401073e-23
  -3.58164087e-23  5.55721967e-23  5.29711941e-23  3.41621024e-23
  -9.05126310e-23 -4.81459431e-23  2.06481530e-23 -3.00153896e-22]
 [-9.43696

In [15]:
p_block = total_station_covariance.diagonal_block("P")

print("Covariance block of point P:")
print(p_block)

print()
print("Coordinate standard deviations of P:")
for parameter in ("P_E", "P_N", "P_H"):
    sigma = total_station_covariance.standard_deviations[parameter]
    print(f"  {parameter}: {sigma:.6f}")

Covariance block of point P:
[[5.26676311e-06 4.07962840e-06 3.36652322e-07]
 [4.07962840e-06 4.48792496e-06 3.06047565e-07]
 [3.36652322e-07 3.06047565e-07 5.45436004e-07]]

Coordinate standard deviations of P:
  P_E: 0.002295
  P_N: 0.002118
  P_H: 0.000739


## 4. Угловая засечка с одной станции тахеометра

Рассмотрим установку тахеометра на известной станции $S$.
Наблюдаются горизонтальные направления:

- на исходную точку `A`;
- на исходную точку `B`;
- на определяемую точку `P`.

Поскольку общий ноль лимба исключается, полезную геометрическую
информацию составляют углы:

$$
\alpha_{SB} - \alpha_{SA},
$$

$$
\alpha_{SP} - \alpha_{SA}.
$$

В данном примере используются только горизонтальные направления.
Высота точки $P$ не определяется и закрепляется отдельно.

Важно: одной станции недостаточно для полноценной надёжной угловой
засечки в произвольной конфигурации. Этот пример демонстрирует
математическую обработку серии направлений и влияние общего нуля лимба,
а не заменяет полноценную сеть с контролем и избыточностью.

In [16]:
angular_network = GeodeticNetwork(
    name="Single-station angular intersection",
    points=(
        make_enh_point("S", 0.0, 0.0, 0.0),
        make_enh_point("A", 0.0, 100.0, 0.0),
        make_enh_point("B", 100.0, 0.0, 0.0),
        make_enh_point("P", 55.0, 65.0, 0.0),
    ),
)

In [21]:
angular_setup = TotalStationSetup(
    name="S_angle_set_01",
    station="S",
    reference_target="A",
    sights=(
        TotalStationSight(
            target="A",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
        ),
        TotalStationSight(
            target="B",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
        ),
        TotalStationSight(
            target="P",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
            slope_distance_standard_deviation=0.003,
        ),
    ),
)

In [22]:
angular_linearized = angular_setup.linearize(angular_network)

print("Labels of reduced angular observations:")
for label in angular_linearized.labels:
    print(f"  {label}")

print()
print("Reduced angular design matrix:")
print(angular_linearized.design_matrix)

print()
print("Reduced angular covariance matrix:")
print(angular_linearized.covariance)

Labels of reduced angular observations:
  S_angle_set_01:A->B:horizontal-angle
  S_angle_set_01:A->P:horizontal-angle
  S_angle_set_01:P:slope-distance

Reduced angular design matrix:
[[ 0.01        0.01        0.         -0.01        0.          0.
   0.         -0.01        0.          0.          0.          0.        ]
 [ 0.00103448  0.00758621  0.         -0.01        0.          0.
   0.          0.          0.          0.00896552 -0.00758621  0.        ]
 [-0.64594224 -0.76338629 -0.          0.          0.          0.
   0.          0.          0.          0.64594224  0.76338629  0.        ]]

Reduced angular covariance matrix:
[[1.88035444e-10 9.40177222e-11 0.00000000e+00]
 [9.40177222e-11 1.88035444e-10 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 9.00000000e-06]]


In [23]:
angular_method = ObservationCovarianceMethod(
    observations=(
        # Фиксируем станцию тахеометра.
        ControlPointObservation.fixed(
            name="S_fixed",
            point_name="S",
            axes=("E", "N", "H"),
            coordinates=(0.0, 0.0, 0.0),
        ),

        # Фиксируем две исходные точки.
        # Они задают геометрию и внешний датум.
        ControlPointObservation.fixed(
            name="A_fixed",
            point_name="A",
            axes=("E", "N", "H"),
            coordinates=(0.0, 100.0, 0.0),
        ),
        ControlPointObservation.fixed(
            name="B_fixed",
            point_name="B",
            axes=("E", "N", "H"),
            coordinates=(100.0, 0.0, 0.0),
        ),

        # Направления с одной установки.
        angular_setup,

        # Горизонтальные направления не определяют высоту P.
        ControlPointObservation.fixed(
            name="P_height_fixed",
            point_name="P",
            axes=("H",),
            coordinates=(0.0,),
        ),
    ),
)

In [24]:
angular_covariance = angular_network.compute_covariance(
    angular_method
)

print_covariance_summary(angular_covariance)

Method: linearized-observations
Dimension: 12
Datum: fixed-control
Minimum eigenvalue: -7.458e-22

Standard deviations:
       S_E: 0.000000
       S_N: -0.000000
       S_H: 0.000000
       A_E: -0.000000
       A_N: 0.000000
       A_H: 0.000000
       B_E: 0.000000
       B_N: 0.000000
       B_H: 0.000000
       P_E: 0.002086
       P_N: 0.002381
       P_H: 0.000000

Covariance matrix:
[[ 2.49159360e-22 -1.67198188e-22  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00 -7.33009281e-23
   0.00000000e+00  1.08410157e-21  9.31594752e-22  0.00000000e+00]
 [-1.67198188e-22 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  7.57637850e-22  2.02431011e-22  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.0000

In [25]:
print("Covariance block of point P:")
print(angular_covariance.diagonal_block("P"))

print()
print("Standard deviations of point P:")
for parameter in ("P_E", "P_N", "P_H"):
    sigma = angular_covariance.standard_deviations[parameter]
    print(f"  {parameter}: {sigma:.6f}")

Covariance block of point P:
[[4.35100973e-06 3.93376100e-06 0.00000000e+00]
 [3.93376100e-06 5.67143300e-06 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00]]

Standard deviations of point P:
  P_E: 0.002086
  P_N: 0.002381
  P_H: 0.000000


## 4. Обратная угловая засечка: свободное станционирование тахеометра

В этом примере тахеометр установлен на определяемой точке \(P\).
Из станции \(P\) наблюдаются три известные опорные точки:

$$
A,\qquad B,\qquad C.
$$

Для каждой опорной точки измеряется горизонтальный отсчёт по лимбу.

Модель отдельного отсчёта:

$$
r_{P\to i} = \alpha_{P\to i} - \omega_P,
$$

где:

- $r_{P\to i}$ — отсчёт по горизонтальному кругу;
- $\alpha_{P\to i}$ — сеточный азимут на опорную точку;
- $\omega_P$ — общая неизвестная ориентировка, или ноль лимба.

`TotalStationSetup` исключает $\omega_P$ преобразованием направлений
в разности относительно визуры на `A`:

$$
\alpha_{PB} - \alpha_{PA},
$$

$$
\alpha_{PC} - \alpha_{PA}.
$$

После этого остаются две независимые угловые связи, определяющие
плановые координаты неизвестной станции:

$$
E_P,\qquad N_P.
$$

Высота $H_P$ не определяется горизонтальными направлениями и
в этом учебном примере закрепляется отдельно.

In [26]:
resection_network = GeodeticNetwork(
    name="Three-point angular resection",
    points=(
        # Три известных опорных пункта.
        make_enh_point("A", 0.0, 100.0, 0.0),
        make_enh_point("B", 120.0, 0.0, 0.0),
        make_enh_point("C", -80.0, -40.0, 0.0),

        # Приближённые координаты неизвестной станции тахеометра.
        # Для априорного ковариационного анализа используются именно
        # приближённые координаты, в которых вычисляются производные.
        make_enh_point("P", 25.0, 20.0, 0.0),
    ),
)

In [27]:
resection_setup = TotalStationSetup(
    name="P_resection_set_01",
    station="P",
    reference_target="A",
    sights=(
        # Визура на A является внутренним ориентиром набора.
        TotalStationSight(
            target="A",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
        ),

        # После исключения нуля лимба будут использованы разности:
        # alpha(P->B) - alpha(P->A)
        TotalStationSight(
            target="B",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
        ),

        # alpha(P->C) - alpha(P->A)
        TotalStationSight(
            target="C",
            horizontal_standard_deviation=arcseconds_to_radians(2.0),
        ),
    ),
)

In [28]:
resection_linearized = resection_setup.linearize(
    resection_network
)

print("Размер матрицы коэффициентов:")
print(resection_linearized.design_matrix.shape)

print()
print("Метки приведённых угловых наблюдений:")
for label in resection_linearized.labels:
    print(f"  {label}")

print()
print("Матрица коэффициентов приведённых направлений:")
print(resection_linearized.design_matrix)

print()
print("Ковариационная матрица приведённых направлений:")
print(resection_linearized.covariance)

Размер матрицы коэффициентов:
(2, 12)

Метки приведённых угловых наблюдений:
  P_resection_set_01:A->B:horizontal-angle
  P_resection_set_01:A->C:horizontal-angle

Матрица коэффициентов приведённых направлений:
[[-0.0113879  -0.00355872  0.         -0.00212202 -0.01007958  0.
   0.          0.          0.          0.01350992  0.01363829  0.        ]
 [-0.0113879  -0.00355872  0.          0.          0.          0.
  -0.00410256  0.00717949  0.          0.01549046 -0.00362077  0.        ]]

Ковариационная матрица приведённых направлений:
[[1.88035444e-10 9.40177222e-11]
 [9.40177222e-11 1.88035444e-10]]


## Выводы

В ноутбуке рассмотрены четыре уровня модели Covarion:

1. **Линейная засечка** использует расстояния как независимые
   геометрические связи.
2. **Полярный способ по азимуту и расстоянию** применим, когда
   направление уже задано относительно внешнего севера.
3. **Установка тахеометра** объединяет визуры одной станции и
   исключает общий ноль лимба через преобразование направлений.
4. **Угловая засечка** демонстрирует, что серия направлений с одной
   станции несёт информацию о взаимных углах, а не об абсолютном
   нуле горизонтального круга.

Важный принцип Covarion:

- `GeodeticPoint` хранит параметры сети;
- наблюдения определяют геометрию и стохастическую модель;
- контрольные точки задают датум;
- `ObservationCovarianceMethod` строит ковариационную матрицу
  координат из линейризованной модели Гаусса–Маркова.

---
---